In [5]:
# ============================================================
# Download PLUG corpus and extract sad Ukrainian sentences
# ============================================================

!pip -q install stanza pandas tqdm gitpython

import re
import subprocess
from pathlib import Path

import pandas as pd
import stanza
from tqdm import tqdm

# ------------------------------------------------------------
# Clone repository
# ------------------------------------------------------------

REPO = "https://github.com/Dandelliony/pluperfect_grac.git"
ROOT = Path("pluperfect_grac")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO], check=True)

# ------------------------------------------------------------
# Download Ukrainian tokenizer
# ------------------------------------------------------------

stanza.download("uk")

nlp = stanza.Pipeline(
    lang="uk",
    processors="tokenize",
    use_gpu=False,
    verbose=False
)

# ------------------------------------------------------------
# Find text files
# ------------------------------------------------------------

files = list(ROOT.rglob("*.txt"))

print(f"Found {len(files)} text files.")

# ------------------------------------------------------------
# Sadness lexicon
# Important: we avoid broad body-part emotional markers because they create too many false positives.
# ------------------------------------------------------------

SAD_WORDS = {
    # grief / sadness
    "горе", "горя", "горем", "горенько",
    "туга", "туги", "тугою",
    "печаль", "печалі", "печаллю", "печальний", "печальна", "печальне",
    "смуток", "смутку", "смутком", "смутний", "смутна", "смутне",
    "сум", "сумно", "сумний", "сумна", "сумне", "сумні",
    "журба", "журби", "журбою", "журитися", "журиться", "журилась", "журився",
    "жаль", "жалю", "жалем", "жалоба", "жалоби", "жалобний", "жалобна",
    "скорбота", "скорботи", "скорбний", "скорбна",

    # crying / tears
    "сльоза", "сльози", "сльозами", "сльозою",
    "ридати", "ридала", "ридав", "ридали", "ридає", "ридають",
    "плакати", "плакала", "плакав", "плакали", "плаче", "плачуть",
    "плач", "плачу", "плачем", "заплакала", "заплакав", "заридав", "заридала",

    # pain / suffering
    "мука", "муки", "мукою",
    "мучитись", "мучиться", "мучилась", "мучився", "мучилися",
    "страждання", "страждати", "страждала", "страждав", "страждає", "страждали",
    "болить", "боліло", "боліти", "болючий", "болюча", "болісний", "болісна",
    "терпіти", "терпіла", "терпів", "терпить", "терпіння",
    "рана", "рани", "ранить", "ранило", "зранений", "зранена",

    # loneliness / abandonment
    "самота", "самотній", "самотня", "самотне", "самотні",
    "одинокий", "одинока", "одиноке", "одинокі",
    "сирота", "сиротою", "сирітський", "сирітська",
    "покинутий", "покинута", "покинув", "покинула", "покинули",
    "забутий", "забута", "забули", "забув",
    "чужий", "чужа", "чужина", "чужині",

    # misfortune / bad fate
    "нещасний", "нещасна", "нещасне", "нещасні",
    "безталанний", "безталанна", "безталанне",
    "безщасний", "безщасна",
    "недоля", "недолі", "недолею",
    "лихо", "лиха", "лихом", "лихий", "лиха",
    "біда", "біди", "бідою", "бідний", "бідна",
    "злидні", "злиднів",

    # despair / hopelessness
    "розпач", "розпачі", "розпачем",
    "відчай", "відчаю", "відчаєм",
    "зневіра", "зневіри", "зневірився", "зневірилась",
    "безнадія", "безнадії", "безнадійний", "безнадійна",
    "марно", "марний", "марна", "марне",
    "даремно", "несила", "нестерпно",

    # loss / death
    "втрата", "втрати", "втратив", "втратила", "втратили",
    "загубив", "загубила", "загубили",
    "помер", "померла", "померли",
    "умер", "умерла", "умерли",
    "смерть", "смерті", "смертю",
    "могила", "могили", "могилою",
    "труна", "труни", "труною",
    "похорон", "поховали", "ховали",

    # hardship
    "тяжко", "тяжкий", "тяжка", "тяжке", "тяжкі",
    "важко", "важкий", "важка", "важке", "важкі",
    "гірко", "гіркий", "гірка", "гірке",
    "страшно", "страшний", "страшна",

    # love suffering
    "розлука", "розлуки", "розлукою",
    "зрада", "зради", "зрадив", "зрадила",
}

SAD_PHRASES = {
    # hard life
    "тяжко мені",
    "важко мені",
    "гірко мені",
    "тяжко жити",
    "важко жити",
    "гірко жити",
    "на світі жити",
    "не маю сили",
    "не маю сил",
    "нема сили",
    "немає сили",
    "не стало сили",
    "несила мені",

    # fate / misfortune
    "нема долі",
    "немає долі",
    "не маю долі",
    "лиха доля",
    "тяжка доля",
    "гірка доля",
    "моя недоля",
    "без долі",
    "ой доле",
    "ой недоле",
    "лихо мені",
    "біда мені",

    # crying / tears
    "сльози ллються",
    "сльози течуть",
    "сльози котяться",
    "обливалася сльозами",
    "обливався сльозами",
    "гірко плакала",
    "гірко плакав",
    "тихо плакала",
    "тихо плакав",
    "не висихали сльози",
    "залилася сльозами",

    # loneliness / abandonment
    "одна одинока",
    "один одинокий",
    "сам собі",
    "сама собі",
    "ніхто не чує",
    "ніхто не бачить",
    "ніхто не знає",
    "ніхто не любить",
    "ніхто не розуміє",
    "покинута всіма",
    "покинутий всіма",
    "на чужині",
    "між чужими людьми",
    "чужі люди",

    # despair / hopelessness
    "нема надії",
    "немає надії",
    "не маю надії",
    "втратила надію",
    "втратив надію",
    "усе пропало",
    "все пропало",
    "марно жити",
    "даремно жити",
    "нема вороття",
    "немає вороття",
    "не вернеться",
    "не повернеться",

    # love suffering / breakup
    "милий покинув",
    "мила покинула",
    "коханий покинув",
    "кохана покинула",
    "тяжка розлука",
    "гірка розлука",
    "зрадив мене",
    "зрадила мене",
    "не любить мене",
    "любові нема",
    "нема любові",

    # death / loss
    "пішов з життя",
    "пішла з життя",
    "лежить у могилі",
    "сира земля",
    "чорна земля",
    "до могили",
    "в домовину",
    "немає живого",
    "нема живого",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

word_pattern = re.compile(r"[а-щьюяґєії'-]+")

def sadness_score(text):
    """
    Returns:
        score: int
        word_hits: list[str]
        phrase_hits: list[str]

    Phrase hits count more because they are usually much better sadness signals.
    """
    text_lower = text.lower()
    words = word_pattern.findall(text_lower)

    word_hits = [w for w in words if w in SAD_WORDS]
    phrase_hits = [p for p in SAD_PHRASES if p in text_lower]

    score = len(word_hits) + 2 * len(phrase_hits)

    return score, word_hits, phrase_hits


2026-06-28 20:50:20 INFO: Downloaded file to /Users/katerynaboguslavska/Library/Caches/stanza/1.13.0/resources/resources.json
2026-06-28 20:50:20 INFO: Downloading default packages for language: uk (Ukrainian) ...
2026-06-28 20:50:20 INFO: File exists: /Users/katerynaboguslavska/Library/Caches/stanza/1.13.0/resources/uk/default.zip
2026-06-28 20:50:21 INFO: Finished downloading models and saved to /Users/katerynaboguslavska/Library/Caches/stanza/1.13.0/resources


Found 98279 text files.


In [6]:
import random

RANDOM_SEED = 42
N_FILES = 3000

random.seed(RANDOM_SEED)
files_experiment = random.sample(files, min(N_FILES, len(files)))

print(f"Using {len(files_experiment)} files for extraction.")


Using 3000 files for extraction.


In [7]:
# ------------------------------------------------------------
# Extract sentences
# ------------------------------------------------------------

rows = []
MIN_SCORE = 2
MIN_SENT_LEN = 40

for file in tqdm(files_experiment):

    try:
        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    if len(text) < 100:
        continue

    doc = nlp(text)

    for sent in doc.sentences:

        sentence = sent.text.strip()

        if len(sentence) < MIN_SENT_LEN:
            continue

        score, word_hits, phrase_hits = sadness_score(sentence)

        if score >= MIN_SCORE:

            rows.append({
                "score": score,
                "sentence": sentence,
                "word_hits": ", ".join(sorted(set(word_hits))),
                "phrase_hits": " | ".join(sorted(set(phrase_hits))),
                "source_file": file.name,
                "path": str(file)
            })

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

if rows:
    df = (
        pd.DataFrame(rows)
          .sort_values("score", ascending=False)
          .drop_duplicates("sentence")
          .reset_index(drop=True)
    )
else:
    df = pd.DataFrame(columns=["score", "sentence", "word_hits", "phrase_hits", "source_file", "path"])

print(f"\nSad sentences found: {len(df):,}")

df.to_csv(
    "sad_sentences.csv",
    index=False,
    encoding="utf-8"
)

display(df.head(30))

print("\nSaved to sad_sentences.csv")


100%|██████████| 3000/3000 [08:20<00:00,  6.00it/s]



Sad sentences found: 1,086


,score,sentence,word_hits,phrase_hits,source_file,path
0,6,"Плакала вона з горя, з злиднів, з турботи і жа...","горя, жалю, злиднів, плакала, туги",,V_nedilu_rano_zilla_kopala.txt,pluperfect_grac/PluG2_texts/O/Olga_Kobilanska/...
1,6,Молодиця плакала та крізь сльози все казала та...,"боліло, горе, горя, плакала, сльози",,Todi_iak_akatsii_tsvily.txt,pluperfect_grac/PluG2_texts/L/Leonid_Paharevsk...
2,6,"І плаче Машука, і плаче віками, струмками без ...","плаче, сльози",сльози течуть,oleksandr_oles_tom2_zneopublikovanogo_kazbek.txt,pluperfect_grac/PluG_texts/O/Oleksandr_Oles/ol...
3,6,На похоронах Кропивницького сльози стискали йо...,"могили, могилою, сльози",до могили,76542.txt,pluperfect_grac/PluG2_texts/Z/Zbruch_Dyskursy_...
4,5,"Нехай вас Бог так не забуде, як ви не забули, ...","забули, покинули, чужині",на чужині,Shevchenko_do_Lazarevskogo_1857_04_05.txt,pluperfect_grac/PluG2_texts/T/Taras_Sevcenko/S...
5,5,"Дівчина плаче; плаче вона не од болю, плаче во...","жаль, плаче",,Dniprova_Chajka_Divchyna_chajka.txt,pluperfect_grac/PluG2_texts/D/Dniprova_Cajka/D...
6,5,"За життя такі далекі собі, такі неподібні, а п...","померли, смерті, чужині",на чужині,Motra.txt,pluperfect_grac/PluG_texts/B/Bogdan_Lepkij/Mot...
7,5,"І в четвертий \nПровела небогу \nАж у поле, до...","могили, покинула, сумно",до могили,Ridna_mova_1937_12_Taras_Shevchenko_Praktychni...,pluperfect_grac/PluG_texts/R/Ridna_mova_1937_1...
8,4,"Обличчя її смутне, з очей ллються гарячі сльоз...","болить, сльози, смутне",,ianovska-liubov-oleksandrivna-smert-makarykhy1...,pluperfect_grac/PluG_texts/L/Lubov_Janovska/ia...
9,4,"Хоч Степан дуже зрадів, побачивши Настю і найс...","жаль, заплакав, страшний, туга",,chaykovskyy-andriy-iakovych-krashche-smert-iak...,pluperfect_grac/PluG2_texts/A/Andrij_Cajkovski...



Saved to sad_sentences.csv


In [8]:
df.shape

(1086, 6)

In [12]:
filtered = pd.read_csv("/Users/katerynaboguslavska/Desktop/Data_Science/ai-engineering/final_project/data/sad_sentences_filtered.csv")
display(filtered.head(30))

,score,sentence,word_hits,phrase_hits,source_file,path
0,6,Молодиця плакала та крізь сльози все казала та...,"боліло, горе, горя, плакала, сльози",NaN,Todi_iak_akatsii_tsvily.txt,pluperfect_grac/PluG2_texts/L/Leonid_Paharevsk...
1,6,"І плаче Машука, і плаче віками, струмками без ...","плаче, сльози",сльози течуть,oleksandr_oles_tom2_zneopublikovanogo_kazbek.txt,pluperfect_grac/PluG_texts/O/Oleksandr_Oles/ol...
2,4,Я оце вже третій рік як пропадаю в неволі – в ...,тяжко,тяжко мені,Shevchenko_do_Bodanskogo_1850_01_03.txt,pluperfect_grac/PluG_texts/T/Taras_Sevcenko/Sh...
3,4,"Знаю, як тобі тяжко думати про те, що ти будеш...",тяжко,тяжко мені,Sonachnyj_promin.txt,pluperfect_grac/PluG_texts/B/Boris_Grincenko/S...
4,5,"Дівчина плаче; плаче вона не од болю, плаче во...","жаль, плаче",NaN,Dniprova_Chajka_Divchyna_chajka.txt,pluperfect_grac/PluG2_texts/D/Dniprova_Cajka/D...
5,6,"Плакала вона з горя, з злиднів, з турботи і жа...","горя, жалю, злиднів, плакала, туги",NaN,V_nedilu_rano_zilla_kopala.txt,pluperfect_grac/PluG2_texts/O/Olga_Kobilanska/...
6,4,"Федір\nТа воно, може, й навчилося б, мовляли, ...",біда,біда мені,Rozumnyj_i_duren.txt,pluperfect_grac/PluG_texts/I/Ivan_Karpenko_Kar...
7,3,"Усе минуле, що не покидало мене і не відпускал...",чужині,на чужині,Batkova_kazka.txt,pluperfect_grac/PluG_texts/I/Ivan_Karpenko_Kar...
8,4,Дайте мені на пам’ять оцей хрест з ваших груде...,"смерть, чужині",на чужині,Batkova_kazka.txt,pluperfect_grac/PluG_texts/I/Ivan_Karpenko_Kar...
9,4,"А як співав їм ""Ой біда мені на тій чужині, що...","біда, чужині",біда мені,73378.txt,pluperfect_grac/PluG2_texts/Z/Zbruch_Dyskursy_...
